# Capacity Allocation

Comparison of metaheuristic algorithms for the revenue-maximizing timetabling
problem. Several mealpy algorithms (GA, DE, PSO, SCA) are run with multiple
seeds; the results are collected into a DataFrame, visualized with box/line
plots, and compared with a Wilcoxon signed-rank test.


## 0. Load Libraries


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from robin.supply.generator.entities import SupplyGenerator
from robin.supply.entities import Supply

from craft import RevenueSimulator, Solution
from craft.mealpy import MealpyTimetabling
from craft.plotter import sns_box_plot, sns_line_plot
from mealpy import FloatVar, GA, DE, PSO, SCA

## 1. Generate Supply




In [2]:
supply_config_path = Path('../configs/supply_generator/supply_data.yaml')
generator_config_path = Path('../configs/supply_generator/config.yaml')
generator_save_path = Path('../data/results/supply_capacity.yaml')

Path('../data/results').mkdir(parents=True, exist_ok=True)

generator = SupplyGenerator.from_yaml(
    path_config_supply=supply_config_path,
    path_config_generator=generator_config_path,
)

generator.generate(
    n_services=25,
    output_path=generator_save_path,
    seed=42,
    progress_bar=True,
    without_conflicts=False,
)

print(f'Generated {len(generator.services)} services')

Generating services all: 100%|██████████| 25/25 [00:00<00:00, 434.15service/s]

Generated 25 services


## 2. Load Supply and Build Problem


In [3]:
supply = Supply.from_yaml(path=str(generator_save_path))
revenue_behavior = RevenueSimulator(supply=supply).simulate_revenue(alpha=2/3)

timetabling = MealpyTimetabling(
    requested_services=supply.services,
    revenue_behavior=revenue_behavior,
    safe_headway=10,
    max_stop_time=10,
)

bounds = [FloatVar(lb=lb, ub=ub) for lb, ub in timetabling.boundaries.real]

problem = {
    'obj_func': timetabling.objective_function,
    'bounds': bounds,
    'minmax': 'max',
    'verbose': False,
}

print(f'Problem with {timetabling.n_services} services and {len(bounds)} real variables')

Problem with 25 services and 81 real variables


## 3. Run Experiments

Each algorithm is run with several seeds to gather statistics. The best
fitness and convergence history are collected for later analysis.


In [4]:
algorithms = {
    'GA':  lambda: GA.BaseGA(epoch=50, pop_size=20, pc=0.9, pm=0.01),
    'DE':  lambda: DE.OriginalDE(epoch=50, pop_size=20, wf=0.5, cr=0.9),
    'PSO': lambda: PSO.OriginalPSO(epoch=50, pop_size=20, c1=1.5, c2=1.5, w=0.7),
    'SCA': lambda: SCA.OriginalSCA(epoch=50, pop_size=20),
}

seeds = [42, 123, 456, 789, 2024]

rows = []
convergence_rows = []

for algo_name, algo_factory in algorithms.items():
    for seed in seeds:
        model = algo_factory()
        model.solve(problem, seed=seed)
        best_fitness = model.g_best.target.fitness
        rows.append({'Algorithm': algo_name, 'Seed': seed, 'Fitness': best_fitness})
        for it, data in enumerate(model.history.list_global_best):
            convergence_rows.append({
                'Algorithm': algo_name,
                'Seed': seed,
                'Iteration': it,
                'Fitness': data.target.fitness,
            })
        print(f'{algo_name} seed={seed}: fitness={best_fitness:.2f}')

df_results = pd.DataFrame(rows)
df_convergence = pd.DataFrame(convergence_rows)

print(f'\nCollected {len(df_results)} runs across {len(algorithms)} algorithms')

2026/11/05 01:17:54 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: BaseGA(epoch=50, pop_size=20, pc=0.9, pm=0.01)
2026/11/05 01:17:54 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 1, Current best: 544.5849615223184, Global best: 544.5849615223184, Runtime: 0.02556 seconds
2026/11/05 01:17:54 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 2, Current best: 515.563576030139, Global best: 544.5849615223184, Runtime: 0.02651 seconds
2026/11/05 01:17:54 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 3, Current best: 515.997383596238, Global best: 544.5849615223184, Runtime: 0.02501 seconds
2026/11/05 01:17:54 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 4, Current best: 516.1182537951979, Global best: 544.5849615223184, Runtime: 0.02569 seconds
2026/11/05 01:17:54 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 5, Current best: 516.2731620375205, Global best: 544.5849615223184, Run

GA seed=42: fitness=587.80


2026/11/05 01:17:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 8, Current best: 518.8995178111676, Global best: 518.9868293551469, Runtime: 0.02588 seconds
2026/11/05 01:17:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 9, Current best: 518.9173717154675, Global best: 518.9868293551469, Runtime: 0.02095 seconds
2026/11/05 01:17:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 10, Current best: 554.321298554213, Global best: 554.321298554213, Runtime: 0.02337 seconds
2026/11/05 01:17:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 11, Current best: 518.9577255071538, Global best: 554.321298554213, Runtime: 0.02231 seconds
2026/11/05 01:17:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 12, Current best: 518.9577255071538, Global best: 554.321298554213, Runtime: 0.02125 seconds
2026/11/05 01:17:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 13

GA seed=123: fitness=554.65


2026/11/05 01:17:57 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 9, Current best: 518.319114402964, Global best: 518.4273281236602, Runtime: 0.02527 seconds
2026/11/05 01:17:57 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 10, Current best: 518.2625996677725, Global best: 518.4273281236602, Runtime: 0.02186 seconds
2026/11/05 01:17:57 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 11, Current best: 518.319114402964, Global best: 518.4273281236602, Runtime: 0.02193 seconds
2026/11/05 01:17:57 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 12, Current best: 518.319114402964, Global best: 518.4273281236602, Runtime: 0.02362 seconds
2026/11/05 01:17:57 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 13, Current best: 518.4187484549132, Global best: 518.4273281236602, Runtime: 0.02271 seconds
2026/11/05 01:17:57 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 

GA seed=456: fitness=555.48


... (30 outputs left out)

2026/11/05 01:18:10 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 10, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01780 seconds
2026/11/05 01:18:10 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 11, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01811 seconds
2026/11/05 01:18:10 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 12, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01739 seconds
2026/11/05 01:18:10 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 13, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01846 seconds
2026/11/05 01:18:10 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 14, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01750 seconds
2026/11/05 01:18:10 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 15, Curr

SCA seed=789: fitness=864.50


2026/11/05 01:18:11 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 12, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01747 seconds
2026/11/05 01:18:11 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 13, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01649 seconds
2026/11/05 01:18:11 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 14, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01726 seconds
2026/11/05 01:18:11 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 15, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01687 seconds
2026/11/05 01:18:11 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 16, Current best: 864.5042191686985, Global best: 864.5042191686985, Runtime: 0.01785 seconds
2026/11/05 01:18:11 PM, INFO, mealpy.math_based.SCA.OriginalSCA: >>>Problem: P, Epoch: 17, Curr

SCA seed=2024: fitness=864.50

Collected 20 runs across 4 algorithms


## 4. Results Summary


In [ ]:
summary = df_results.groupby('Algorithm')['Fitness'].agg(['mean', 'std', 'min', 'max'])
summary = summary.sort_values('mean', ascending=False)
summary.columns = ['Mean', 'Std', 'Min', 'Max']
summary

## 5. Box Plot


In [ ]:
sns_box_plot(
    df=df_results,
    x_data='Algorithm',
    y_data='Fitness',
    title='Algorithm Comparison (Revenue)',
    x_label='Algorithm',
    y_label='Best Fitness',
)

## 6. Convergence Plot


In [ ]:
sns_line_plot(
    df=df_convergence,
    x_data='Iteration',
    y_data='Fitness',
    hue='Algorithm',
    title='Convergence by Algorithm',
    x_label='Iteration',
    y_label='Best Fitness',
    x_limit=(0, 50),
)

## 7. Statistical Comparison (Wilcoxon)

Pairwise Wilcoxon signed-rank test between the best algorithm and the rest,
on the per-seed fitness values.


In [ ]:
best_algo = summary.index[0]
best_values = df_results[df_results['Algorithm'] == best_algo]['Fitness'].values

print(f'Best algorithm: {best_algo} (mean={summary.loc[best_algo, "Mean"]:.2f})')
print()
for algo in algorithms:
    if algo == best_algo:
        continue
    other_values = df_results[df_results['Algorithm'] == algo]['Fitness'].values
    stat, p_value = stats.wilcoxon(best_values, other_values)
    sig = '*' if p_value < 0.05 else 'ns'
    print(f'{best_algo} vs {algo}: W={stat:.1f}, p={p_value:.4f} ({sig})')

## 8. Best Solution


In [ ]:
best_row = df_results.loc[df_results['Fitness'].idxmax()]
best_algo_name = best_row['Algorithm']
best_seed = best_row['Seed']

model = algorithms[best_algo_name]()
model.solve(problem, seed=best_seed)

best_position = model.g_best.solution
schedule = timetabling.get_heuristic_schedule()
solution = Solution(real=best_position, discrete=schedule)

updated_services = timetabling.update_supply(str(generator_save_path), solution)
print(f'Best solution: {best_algo_name} seed={best_seed}, fitness={best_row["Fitness"]:.2f}')
print(f'Scheduled services: {sum(schedule)}/{len(schedule)}')
print(f'Updated supply contains {len(updated_services)} services')